### Experiment Tracking of Params, Artifacts, Metrics and Model with MLFLOW

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split

In [2]:
# read the data and get 10 samples of the data

chicago_df = pd.read_csv("chicago_data.csv")
chicago_df.head()

,person_id,person_type,crash_record_id,vehicle_id_x,crash_date,city,state,zipcode,sex,age,...,location__type,_acomputed_region_rpca_8um6,hit_and_run_i,private_property_i,intersection_related_i,crash_date_est_i,dooring_i,statements_taken_i,photos_taken_i,work_zone_i
0,O2073682,DRIVER,aa6d90de5635ad05fb7b169b2346b67482afe6f2cb1af0...,1976830.0,2025-05-29 20:35:00-05:00,CHICAGO,IL,60619.0,F,37.0,...,Point,19.0,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown
1,O2073682,DRIVER,aa6d90de5635ad05fb7b169b2346b67482afe6f2cb1af0...,1976830.0,2025-05-29 20:35:00-05:00,CHICAGO,IL,60619.0,F,37.0,...,Point,19.0,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown
2,O2073683,DRIVER,aa6d90de5635ad05fb7b169b2346b67482afe6f2cb1af0...,1976831.0,2025-05-29 20:35:00-05:00,HIGHLAND,IN,46322.0,M,28.0,...,Point,19.0,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown
3,O2073683,DRIVER,aa6d90de5635ad05fb7b169b2346b67482afe6f2cb1af0...,1976831.0,2025-05-29 20:35:00-05:00,HIGHLAND,IN,46322.0,M,28.0,...,Point,19.0,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown
4,O2073686,DRIVER,8c11a8d7d0c683a997104ce7149e8bd7c393c3b7b5a818...,1976833.0,2025-05-29 20:07:00-05:00,CICERO,IL,60804.0,M,22.0,...,Point,7.0,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown


In [3]:
chicago_df.columns

Index(['person_id', 'person_type', 'crash_record_id', 'vehicle_id_x',
       'crash_date', 'city', 'state', 'zipcode', 'sex', 'age',
       ...
       'location__type', '_acomputed_region_rpca_8um6', 'hit_and_run_i',
       'private_property_i', 'intersection_related_i', 'crash_date_est_i',
       'dooring_i', 'statements_taken_i', 'photos_taken_i', 'work_zone_i'],
      dtype='object', length=124)

* get datetime, categorical and numerical variables from the dataframe columns

In [14]:
numerical_variables = [
    'vehicle_id_x', 'age', 'zipcode', 'seat_no', 'ems_run_no', 'crash_unit_id', 'unit_no', 
    'vehicle_id_y', 'vehicle_year', 'occupant_cnt', 'cmv_id', 'usdot_no', 'gvwr', 'num_passengers', 'posted_speed_limit', 
    'street_no', 'beat_of_occurrence', 'num_units','injuries_total', 'injuries_fatal', 'injuries_incapacitating', 'injuries_non_incapacitating', 
    'injuries_reported_not_evident', 'injuries_no_indication', 'injuries_unknown', 'crash_hour', 'latitude', 'longitude', 
    '_acomputed_region_rpca_8um6' 
]

categorical_variables = [
    'person_id', 'crash_record_id', 'person_type', 'city', 'state', 'sex', 'drivers_license_state', 'drivers_license_class', 'safety_equipment', 
    'airbag_deployed', 'ejection', 'injury_classification', 'driver_action', 'driver_vision', 'physical_condition',
    'bac_result', 'hospital', 'ems_agency', 'pedpedal_action', 'pedpedal_visibility', 'pedpedal_location',
    'unit_type', 'make', 'model', 'lic_plate_state', 'vehicle_defect', 'vehicle_type', 'vehicle_use', 'travel_direction',
    'maneuver', 'area_01_i', 'area_02_i', 'first_contact_point', 'area_07_i', 'area_08_i', 'area_06_i',
    'area_05_i', 'cmrc_veh_i', 'towed_i','towed_by', 'towed_to', 'area_12_i', 'area_99_i', 'commercial_src', 'carrier_name',
    'carrier_state', 'carrier_city', 'hazmat_present_i', 'hazmat_report_i', 'mcs_report_i', 'hazmat_vio_cause_crash_i',
    'mcs_vio_cause_crash_i', 'vehicle_config', 'cargo_body_type', 'load_type', 'hazmat_out_of_service_i', 'mcs_out_of_service_i',
    'area_09_i', 'area_10_i', 'area_11_i', 'area_03_i', 'area_04_i', 'area_00_i', 'traffic_control_device', 'device_condition', 
    'weather_condition', 'lighting_condition', 'first_crash_type', 'trafficway_type', 'alignment', 'roadway_surface_cond', 'road_defect',
    'report_type', 'crash_type', 'intersection_related_i', 'damage', 'prim_contributory_cause', 'sec_contributory_cause', 'street_direction',
    'street_name', 'most_severe_injury', 'crash_day_of_week', 'crash_month', 'location__type', 'hit_and_run_i', 'statements_taken_i', 
    'private_property_i', 'crash_date_est_i', 'dooring_i', 'work_zone_i', 'photos_taken_i'
]

datetime_variables = ['crash_date', 'date_police_notified']

* converting objects to datetime datatypes

In [18]:
chicago_df[datetime_variables].dtypes

crash_date              object
date_police_notified    object
dtype: object

In [20]:
for col in datetime_variables:
    chicago_df[col] = pd.to_datetime(chicago_df[col], errors='coerce')

chicago_df[datetime_variables].dtypes

crash_date              datetime64[ns, UTC-05:00]
date_police_notified    datetime64[ns, UTC-05:00]
dtype: object

* datatypes of numerical and categorical columns

In [31]:
# chicago_df[numerical_variables].dtypes

# chicago_df[categorical_variables].dtypes

#### Setting Up The Validation Framework

In [28]:
# Get Test data:  test_size=0.2, which is the 20% of full dataset

chicago_df_fulltrain, chicago_df_test = train_test_split(chicago_df, test_size=0.2, random_state=1)

In [29]:
# Get Train and Validation data: 20% Validation off the 80% Train == 20/80 == 0.25

chicago_df_train, chicago_df_val = train_test_split(chicago_df_fulltrain, test_size=0.25, random_state=1)

In [32]:
# reset indexes of each dataframe

chicago_df_fulltrain = chicago_df_fulltrain.reset_index(drop=True)
chicago_df_train = chicago_df_train.reset_index(drop=True)
chicago_df_val = chicago_df_val.reset_index(drop=True)
chicago_df_test = chicago_df_test.reset_index(drop=True)

In [41]:
# target values for train, val and test dataframes in numpy array

y_train = chicago_df_train[['latitude', 'longitude']].values
y_val = chicago_df_val[['latitude', 'longitude']].values
y_test = chicago_df_test[['latitude', 'longitude']].values

In [45]:
# delete target variables from each dataframe

chicago_df_train.drop(columns=['latitude', 'longitude'], inplace=True)
chicago_df_val.drop(columns=['latitude', 'longitude'], inplace=True)
chicago_df_test.drop(columns=['latitude', 'longitude'], inplace=True)

In [53]:
len(chicago_df)

2107

In [54]:
len(chicago_df_test) + len(chicago_df_train) + len(chicago_df_val)

2107